# atmacup20 1st place solution
[1位解法](https://www.guruguru.science/competitions/27/discussions/960838f8-36ee-4002-9992-3861c37b7d62/)のシングルモデルを再現するコードを共有します。

- 手元のRTX3090環境で実行時間約80min(1fold約15min)
- CV:0.69525、Public:0.6922、Private:0.7191<br>※なぜか乱数固定ができておらず、実行のたびに若干値がブレます

In [ ]:
# ============================================
# 必要なライブラリのインポート
# ============================================
import os
import random
from pathlib import Path
from typing import Optional, Union
import datetime
import shutil

## 数値計算・機械学習
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

## Transformers (BERT関連)
from transformers import (
    AutoTokenizer,  # トークナイザの自動読み込み
    DataCollatorWithPadding,  # バッチ作成時の動的パディング
    EvalPrediction,  # 評価予測の型定義
    ModernBertForSequenceClassification,  # Modern BERTの分類モデル
    PreTrainedTokenizerBase,  # トークナイザの基底クラス
    Trainer,  # 学習・評価を実行するTrainerクラス
    TrainingArguments,  # 学習設定
)
from transformers.modeling_outputs import SequenceClassifierOutput  # モデル出力の型定義

## データ処理
from datasets import Dataset  # HuggingFace Datasets
import polars as pl  # 高速データフレーム処理ライブラリ
from tqdm.auto import tqdm  # プログレスバー

## ユーティリティ
from box import Box  # 辞書をオブジェクト形式でアクセス可能にする
import yaml  # YAML設定ファイル読み込み

In [ ]:
# ============================================
# 実験設定（YAML形式）
# ============================================
CONFIG = """
## パス設定
input_dir: ../../data/raw/input  # 入力データディレクトリ（環境に応じて変更）
work_dir: ./  # 作業ディレクトリ（モデル保存先など）

## 交差検証設定
n_splits: 3  # Foldの数（3-fold CV）

## モデル設定
model: sbintuitions/modernbert-ja-30m  # 日本語Modern BERTモデル（30Mパラメータ）
max_length: 1024  # 最大トークン数

## 最適化設定
optim_type: adamw_torch  # オプティマイザ（PyTorch実装のAdamW）
per_device_train_batch_size: 8  # GPUあたりの物理バッチサイズ
gradient_accumulation_steps: 8  # 勾配累積ステップ数（実質バッチサイズ=8×8=64）
per_device_eval_batch_size: 16  # 評価時のバッチサイズ
eval_steps: 50  # 評価・保存を行う間隔（ステップ数）
max_steps: 1000  # 最大学習ステップ数
max_grad_norm: 10.0  # 勾配クリッピングの閾値

## 学習率スケジューラ設定
lr: 2.0e-5  # 学習率（BERTの標準的なファインチューニング学習率）
weight_decay: 0.01  # L2正則化の重み減衰
warmup_steps: 25  # ウォームアップステップ数（0から学習率を徐々に上げる）
lr_scheduler_type: cosine  # スケジューラタイプ（Cosineアニーリング）

## 分類ヘッド設定
classifier_pooling: cls  # プーリング方法（cls: [CLS]トークンを使用、mean: 平均プーリング）

## その他
seed: 42  # 乱数シード
exp_name: atma20-bert-001  # 実験名（出力ファイル名に使用）

"""

# YAML文字列を辞書として読み込み、Box形式に変換（ドット記法でアクセス可能）
config = Box(yaml.safe_load(CONFIG))

In [ ]:
# ============================================
# データスキーマ定義
# ============================================
# Polarsで読み込む際の各カラムのデータ型を明示的に指定

## Udemy動画研修の活動ログデータ
UDEMY_ACTIVITY_SCHEMA = {
    "社員番号": pl.Utf8,  # 社員ID（文字列）
    "コースID": pl.Int64,  # コースの一意ID
    "コースタイトル": pl.Utf8,  # コース名
    "レクチャーもしくはクイズ": pl.Utf8,  # レクチャーかクイズかの区別
    "レクチャー/クイズID": pl.Int64,  # レクチャー/クイズの一意ID
    "レクチャー/クイズの題名": pl.Utf8,  # 題名
    "開始日": pl.Datetime,  # 学習開始日時
    "終了日": pl.Datetime,  # 学習終了日時
    "推定完了率%": pl.Float64,  # 完了率（パーセンテージ）
    "最終結果（クイズの場合）": pl.Float64,  # クイズの得点
    "マーク済み修了": pl.Boolean,  # 完了フラグ
    "コースカテゴリー": pl.Utf8,  # カテゴリ（例: ネットワーク、プログラミングなど）
}

## DX研修データ
DX_SCHEMA = {
    "社員番号": pl.Utf8,  # 社員ID
    "研修実施日": pl.Datetime,  # 研修実施日時
    "研修カテゴリ": pl.Utf8,  # 研修カテゴリ（例: リテラシー_DX基礎など）
    "研修名": pl.Utf8,  # 研修名
}

## 人事研修データ
HR_SCHEMA = {
    "社員番号": pl.Utf8,  # 社員ID
    "カテゴリ": pl.Utf8,  # 研修カテゴリ
    "研修名": pl.Utf8,  # 研修名
    "実施日": pl.Utf8,  # 実施日（文字列形式、後で変換）
}

## 月別残業時間データ
OVERTIME_WORK_BY_MONTH_SCHEMA = {
    "社員番号": pl.Utf8,  # 社員ID
    "date": pl.Datetime,  # 年月
    "hours": pl.Float64,  # 残業時間（時間単位）
}

In [ ]:
# ============================================
# ユーティリティ関数
# ============================================

def seed_everything(seed: int = 42, deterministic: bool = False):
    """
    乱数シードを固定して再現性を確保する関数
    
    Args:
        seed: 乱数シード値
        deterministic: cudnnの決定的動作を有効にするか（遅くなる可能性あり）
    """
    random.seed(seed)  # Python標準のrandom
    np.random.seed(seed)  # NumPy
    os.environ["PYTHONHASHSEED"] = str(seed)  # ハッシュ値の固定
    torch.manual_seed(seed)  # PyTorch CPU
    torch.cuda.manual_seed(seed)  # PyTorch GPU（単一GPU）
    torch.backends.cudnn.deterministic = deterministic  # cuDNNの決定的動作


def compute_metrics(eval_preds: EvalPrediction) -> dict:
    """
    評価指標を計算する関数（Trainerに渡す）
    
    Args:
        eval_preds: モデルの予測結果と正解ラベル
        
    Returns:
        評価指標の辞書（ここではROC-AUCスコア）
    """
    ## モデル出力を取得
    logits = eval_preds.predictions  # モデルの生出力（logits）shape: [N, 2]
    labels = eval_preds.label_ids  # 正解ラベル shape: [N]
    
    ## logitsをSoftmaxで確率に変換し、クラス1（正例）の確率を取得
    probs = torch.from_numpy(logits).float().softmax(-1).numpy()[:, 1]
    
    ## ROC-AUCスコアを計算
    auc = roc_auc_score(labels, probs)
    
    return {"auc": auc}


def make_cv(df: pl.DataFrame, n_splits: int, seed: int) -> pl.DataFrame:
    """
    Stratified Group K-Foldで交差検証のFoldを作成
    
    - Stratified: targetの分布を各Foldで均等に保つ
    - Group: 同じ社員番号が異なるFoldに分かれないようにする
    
    Args:
        df: データフレーム（targetと社員番号を含む）
        n_splits: Fold数
        seed: 乱数シード
        
    Returns:
        Fold番号が追加されたデータフレーム
    """
    ## Fold番号を格納する配列を初期化
    folds = np.zeros(len(df))
    kfold = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    ## 各Foldに分割
    for fold, (train_index, valid_index) in enumerate(
        kfold.split(df, df.get_column("target"), df.get_column("社員番号"))
    ):
        folds[valid_index] = fold
    
    ## Fold番号を新しいカラムとして追加
    return df.with_columns(pl.Series(folds, dtype=pl.Int64).alias("fold"))

In [ ]:
# ============================================
# 乱数シード固定（再現性のため）
# ============================================
seed_everything(config.seed, deterministic=True)

In [ ]:
def preprocess_data(
    udemy_activity_df: pl.DataFrame,
    dx_df: pl.DataFrame,
    hr_df: pl.DataFrame,
    overtime_work_by_month_df: pl.DataFrame,
    position_history_df: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame, dict]:
    udemy_activity_df = udemy_activity_df.with_columns(
        (
            pl.when(
                (pl.col("終了日") >= datetime.datetime(2022, 4, 1))
                & (pl.col("終了日") < datetime.datetime(2023, 4, 1))
            )
        )
        .then(pl.lit(2022))
        .when(
            (pl.col("終了日") >= datetime.datetime(2023, 4, 1))
            & (pl.col("終了日") < datetime.datetime(2024, 4, 1))
        )
        .then(pl.lit(2023))
        .when(
            (pl.col("終了日") >= datetime.datetime(2024, 4, 1))
            & (pl.col("終了日") < datetime.datetime(2025, 4, 1))
        )
        .then(pl.lit(2024))
        .when(
            (pl.col("終了日") >= datetime.datetime(2025, 4, 1))
            & (pl.col("終了日") < datetime.datetime(2026, 4, 1))
        )
        .then(pl.lit(2025))
        .alias("year")
    )
    dx_df = dx_df.with_columns(
        (
            pl.when(
                (pl.col("研修実施日") >= datetime.datetime(2022, 4, 1))
                & (pl.col("研修実施日") < datetime.datetime(2023, 4, 1))
            )
        )
        .then(pl.lit(2022))
        .when(
            (pl.col("研修実施日") >= datetime.datetime(2023, 4, 1))
            & (pl.col("研修実施日") < datetime.datetime(2024, 4, 1))
        )
        .then(pl.lit(2023))
        .when(
            (pl.col("研修実施日") >= datetime.datetime(2024, 4, 1))
            & (pl.col("研修実施日") < datetime.datetime(2025, 4, 1))
        )
        .then(pl.lit(2024))
        .alias("year")
    )
    hr_df = (
        hr_df.with_columns(
            pl.col("実施日")
            .str.extract(r"(\d{4}[-/]\d{1,2}[-/]\d{1,2})")
            .str.to_datetime()
            .alias("date")
        )
        .unique()
        .with_columns(
            (
                pl.when(
                    (pl.col("date") >= datetime.datetime(2022, 4, 1))
                    & (pl.col("date") < datetime.datetime(2023, 4, 1))
                )
            )
            .then(pl.lit(2022))
            .when(
                (pl.col("date") >= datetime.datetime(2023, 4, 1))
                & (pl.col("date") < datetime.datetime(2024, 4, 1))
            )
            .then(pl.lit(2023))
            .when(
                (pl.col("date") >= datetime.datetime(2024, 4, 1))
                & (pl.col("date") < datetime.datetime(2025, 4, 1))
            )
            .then(pl.lit(2024))
            .when(
                (pl.col("date") >= datetime.datetime(2025, 4, 1))
                & (pl.col("date") < datetime.datetime(2026, 4, 1))
            )
            .then(pl.lit(2025))
            .alias("year")
        )
    )
    position_history_df = position_history_df.with_columns(
        (pl.col("year") + 2000).alias("year")
    )

    return (
        udemy_activity_df,
        dx_df,
        hr_df,
        overtime_work_by_month_df,
        position_history_df,
    )


def process_data(
    df: pl.DataFrame,
    udemy_df: pl.DataFrame,
    dx_df: pl.DataFrame,
    hr_df: pl.DataFrame,
    overtime_df: pl.DataFrame,
    position_df: pl.DataFrame,
) -> pl.DataFrame:
    prompts = {}
    employee_ids = df.get_column("社員番号").unique(maintain_order=True).to_list()
    for employee_id in tqdm(employee_ids, desc="Processing Prompts"):
        udemy_activity = udemy_df.filter(pl.col("社員番号") == employee_id)
        dx = dx_df.filter(pl.col("社員番号") == employee_id)
        hr = hr_df.filter(pl.col("社員番号") == employee_id)
        overtime = overtime_df.filter(pl.col("社員番号") == employee_id)
        position = position_df.filter(pl.col("社員番号") == employee_id)

        prompt = []
        years = [2022, 2023, 2024]

        for year in years:
            prompt.append(f"{year}年")

            # position
            position_year = position.filter(pl.col("year") == year)
            if position_year.is_empty():
                prompt.append("情報なし")
                prompt.append("")
                continue

            assert len(position_year) == 1, "Multiple positions found for the same year"

            prompt.append(
                f"{position_year.get_column('勤務区分')[0]}({position_year.get_column('役職')[0]})"
            )

            # overtime
            overtime_year = overtime.filter(pl.col("date").dt.year() == year)
            avg_hours = overtime_year.get_column("hours").mean()
            max_hours = overtime_year.get_column("hours").max()
            min_hours = overtime_year.get_column("hours").min()
            avg_hours_5 = round(avg_hours / 5) * 5  # Round to nearest 5
            min_hours = round(min_hours / 5) * 5  # Round to nearest 5
            max_hours = round(max_hours / 5) * 5  # Round to nearest 5
            prompt.append(f"平均残業時間: 約{avg_hours_5}時間")
            prompt.append(f"最大残業時間: 約{max_hours}時間")
            prompt.append(f"最小残業時間: 約{min_hours}時間")

            # dx
            prompt.append("---")
            prompt.append("DX研修")
            dx_year = dx.filter(pl.col("year") == year)
            if not dx_year.is_empty():
                dx_year_head = (
                    (
                        dx_year.group_by("研修カテゴリ")
                        .len()
                        .sort("研修カテゴリ", descending=True)[:5]
                    )
                    .select(
                        (
                            pl.col("研修カテゴリ").str.replace_all("_", " ")
                            + ": "
                            + pl.col("len").cast(pl.Utf8)
                            + "回"
                        ).alias("dx_prompt")
                    )
                    .to_series()
                    .to_list()
                )
                prompt.extend(dx_year_head)

            # hr
            prompt.append("---")
            prompt.append("人事研修")
            hr_year = hr.filter(pl.col("year") == year)
            if not hr_year.is_empty():
                hr_year_head = (
                    hr_year.group_by("カテゴリ")
                    .len()
                    .sort("len", descending=True)
                    .with_columns((pl.col("カテゴリ")).alias("hr_prompt"))
                    .get_column("hr_prompt")
                    .to_list()
                )
                prompt.extend(hr_year_head)

            # udemy_activity
            prompt.append("---")
            prompt.append("動画研修")
            udemy_year = udemy_activity.filter(
                pl.col("year") == year,
                pl.col("マーク済み修了"),
                pl.col("コースカテゴリー").is_not_null(),
            )
            if not udemy_year.is_empty():
                udemy_year_head = (
                    (
                        udemy_year.group_by("コースカテゴリー")
                        .len()
                        .sort("コースカテゴリー", descending=True)[:5]
                    )
                    .select(
                        (
                            pl.col("コースカテゴリー")
                            + ": "
                            + pl.col("len").cast(pl.Utf8)
                            + "回"
                        ).alias("udemy_prompt")
                    )
                    .to_series()
                    .to_list()
                )
                prompt.extend(udemy_year_head)

            prompt.append("")

        prompts[employee_id] = "\n".join(prompt).strip()

    df = df.with_columns(
        pl.col("社員番号").alias("employee_id"),
        pl.col("category").alias("category"),
        (pl.col("category") + "\n" + pl.col("社員番号").replace(prompts)).alias("text"),
        pl.col("target").alias("labels"),
    )

    return df.select("employee_id", "category", "text", "labels", "fold")

In [ ]:
# ============================================
# データ読み込みとプロンプト作成
# ============================================

## パス設定
DATA_DIR = Path(config.input_dir)
WORK_DIR = Path(config.work_dir)
# 作業ディレクトリをクリーンアップして再作成
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(exist_ok=True)

## データ読み込み
train_df = pl.read_csv(DATA_DIR / "train.csv")
dx_df = pl.read_csv(DATA_DIR / "dx.csv", schema=DX_SCHEMA)
hr_df = pl.read_csv(DATA_DIR / "hr.csv", schema=HR_SCHEMA)
overtime_work_by_month_df = pl.read_csv(
    DATA_DIR / "overtime_work_by_month.csv", schema=OVERTIME_WORK_BY_MONTH_SCHEMA
)
udemy_activity_df = pl.read_csv(
    DATA_DIR / "udemy_activity.csv", schema=UDEMY_ACTIVITY_SCHEMA
)
position_history_df = pl.read_csv(DATA_DIR / "position_history.csv")

## 交差検証のFold作成
# Stratified Group K-Fold: 社員ごとにグループ化し、targetの分布を保つ
train_df = make_cv(train_df, n_splits=config.n_splits, seed=config.seed)

## データ前処理（年度カラムの追加）
(udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df) = (
    preprocess_data(
        udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df
    )
)

## プロンプト作成（テーブルデータ→テキスト変換）
# 各社員の研修履歴を構造化テキストに変換
train_data = process_data(
    train_df,
    udemy_activity_df,
    dx_df,
    hr_df,
    overtime_work_by_month_df,
    position_history_df,
)
# 作成したデータを保存
train_data.write_csv(WORK_DIR / "train_data.csv")

In [ ]:
# ===========================
# データ読み込みとプロンプト作成
# ===========================

# パス設定
DATA_DIR = Path(config.input_dir)
WORK_DIR = Path(config.work_dir)
# 作業ディレクトリをクリーンアップして再作成
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(exist_ok=True)

# ===== データ読み込み =====
train_df = pl.read_csv(DATA_DIR / "train.csv")
dx_df = pl.read_csv(DATA_DIR / "dx.csv", schema=DX_SCHEMA)
hr_df = pl.read_csv(DATA_DIR / "hr.csv", schema=HR_SCHEMA)
overtime_work_by_month_df = pl.read_csv(
    DATA_DIR / "overtime_work_by_month.csv", schema=OVERTIME_WORK_BY_MONTH_SCHEMA
)
udemy_activity_df = pl.read_csv(
    DATA_DIR / "udemy_activity.csv", schema=UDEMY_ACTIVITY_SCHEMA
)
position_history_df = pl.read_csv(DATA_DIR / "position_history.csv")

# ===== 交差検証のFold作成 =====
# Stratified Group K-Fold: 社員ごとにグループ化し、targetの分布を保つ
train_df = make_cv(train_df, n_splits=config.n_splits, seed=config.seed)

# ===== データ前処理（年度カラムの追加） =====
(udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df) = (
    preprocess_data(
        udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df
    )
)

# ===== プロンプト作成（テーブルデータ→テキスト変換） =====
# 各社員の研修履歴を構造化テキストに変換
train_data = process_data(
    train_df,
    udemy_activity_df,
    dx_df,
    hr_df,
    overtime_work_by_month_df,
    position_history_df,
)
# 作成したデータを保存
train_data.write_csv(WORK_DIR / "train_data.csv")

In [ ]:
train_data.head(1)

In [ ]:
print(train_data['text'][0])

# ============================================
# トークナイザの読み込みと保存
# ============================================
# Modern BERTの日本語トークナイザを読み込み
tokenizer = AutoTokenizer.from_pretrained(config.model, trust_remote_code=True)
# 後で再利用するために保存
tokenizer.save_pretrained(WORK_DIR / "tokenizer")

In [ ]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model, trust_remote_code=True)
tokenizer.save_pretrained(WORK_DIR / "tokenizer")

In [ ]:
# ============================================
# エンコーディング関数
# ============================================
def encode_prompt(
    batch: dict, tokenizer: PreTrainedTokenizerBase, max_length: int
) -> dict:
    """
    テキストをトークン化する関数
    
    Args:
        batch: テキストを含むバッチ
        tokenizer: トークナイザ
        max_length: 最大トークン数
        
    Returns:
        トークン化されたデータ（input_ids, attention_mask等）
    """
    ## テキストをトークン化
    encoded = tokenizer(
        batch["text"], 
        truncation=True,  # max_lengthを超えたら切り詰める
        padding=False,  # パディングはDataCollatorで後で行う（効率的）
        max_length=max_length
    )
    ## エンコード結果にラベルとFold情報を追加
    return {**encoded, "labels": batch["labels"], "fold": batch["fold"]}

In [ ]:
# ============================================
# データセット変換とトークン化
# ============================================

## Polars DataFrame → HuggingFace Dataset形式に変換
train_ds = Dataset.from_polars(train_data)

## テキストをトークン化
# batched=True: バッチ処理で高速化
# remove_columns: textカラムは不要なので削除（トークン化後）
train_ds = train_ds.map(
    lambda batch: encode_prompt(batch, tokenizer, max_length=config.max_length),
    batched=True,
    remove_columns=["text"],
)

In [ ]:
train_ds

# Modern Bert

In [ ]:
# ============================================
# カスタムModern BERT分類器
# ============================================
class ModernBertClassifier(ModernBertForSequenceClassification):
    """
    Modern BERTをベースにしたカスタム分類器
    プーリング方法（CLSまたはMean）を選択可能
    """
    
    def __init__(self, config):
        super().__init__(config)

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,  # トークンID [batch_size, seq_len]
        attention_mask: Optional[torch.Tensor] = None,  # アテンションマスク [batch_size, seq_len]
        sliding_window_mask: Optional[torch.Tensor] = None,  # スライディングウィンドウマスク
        position_ids: Optional[torch.Tensor] = None,  # 位置ID
        inputs_embeds: Optional[torch.Tensor] = None,  # 埋め込みベクトル（直接入力の場合）
        labels: Optional[torch.Tensor] = None,  # 正解ラベル [batch_size]
        indices: Optional[torch.Tensor] = None,  # インデックス
        cu_seqlens: Optional[torch.Tensor] = None,  # 累積シーケンス長
        max_seqlen: Optional[int] = None,  # 最大シーケンス長
        batch_size: Optional[int] = None,  # バッチサイズ
        seq_len: Optional[int] = None,  # シーケンス長
        output_attentions: Optional[bool] = None,  # アテンション重みを出力するか
        output_hidden_states: Optional[bool] = None,  # 隠れ層を出力するか
        return_dict: Optional[bool] = None,  # 辞書形式で返すか
        **kwargs,
    ) -> Union[tuple[torch.Tensor], SequenceClassifierOutput]:
        """
        順伝播処理
        
        Returns:
            SequenceClassifierOutput: 損失、logits、隠れ層、アテンション重みを含む
        """
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        ## BERT Encoder: 全トークンをエンコード
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sliding_window_mask=sliding_window_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            indices=indices,
            cu_seqlens=cu_seqlens,
            max_seqlen=max_seqlen,
            batch_size=batch_size,
            seq_len=seq_len,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        # 最終層の隠れ状態 shape: [batch_size, seq_len, hidden_size]
        last_hidden_state = outputs[0]

        ## プーリング戦略の選択
        if self.config.classifier_pooling == "cls":
            ### CLSプーリング: [CLS]トークン（先頭）の埋め込みを使用
            # shape: [batch_size, hidden_size]
            last_hidden_state = last_hidden_state[:, 0]
        elif self.config.classifier_pooling == "mean":
            ### Meanプーリング: 全トークンの平均（attention_maskを考慮）
            # padding部分を除外して平均を取る
            last_hidden_state = (last_hidden_state * attention_mask.unsqueeze(-1)).sum(
                dim=1
            ) / attention_mask.sum(dim=1, keepdim=True)

        ## 分類ヘッド
        pooled_output = self.head(last_hidden_state)  # 中間層（線形変換）
        pooled_output = self.drop(pooled_output)  # Dropout（過学習防止）
        logits = self.classifier(pooled_output)  # 最終出力層 shape: [batch_size, num_labels]

        ## 損失計算
        loss = None
        if labels is not None:
            loss_fct = torch.nn.CrossEntropyLoss()  # 多クラス分類用の損失関数
            # logitsとlabelsから損失を計算
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        ## 戻り値の形式
        if not return_dict:
            output = (logits,)
            return ((loss,) + output) if loss is not None else output

        # 辞書形式で返す（推奨）
        return SequenceClassifierOutput(
            loss=loss,  # 損失値
            logits=logits,  # 予測値（確率変換前）
            hidden_states=outputs.hidden_states,  # 各層の隠れ状態
            attentions=outputs.attentions,  # アテンション重み
        )

In [ ]:
# ============================================
# 3-Fold交差検証による学習
# ============================================
oof_predictions = []  # Out-of-Fold予測を格納するリスト

for fold in range(config.n_splits):
    print(f"### Fold {fold + 1}/{config.n_splits} ###")

    ## データ分割
    # 現在のFold以外を学習データ、現在のFoldを検証データとする
    tr_ds = train_ds.filter(lambda example: example["fold"] != fold)
    val_ds = train_ds.filter(lambda example: example["fold"] == fold)

    ## モデルの初期化
    # 事前学習済みのModern BERTモデルを読み込み（毎Fold初期化）
    model = ModernBertClassifier.from_pretrained(
        config.model,
        classifier_pooling=config.classifier_pooling,  # CLSプーリング
        num_labels=2,  # 二値分類（関心あり/なし）
        torch_dtype=torch.bfloat16,  # BFloat16で高速化
        device_map="auto",  # 自動的にGPUに配置
    )

    ## 学習設定
    training_args = TrainingArguments(
        output_dir=WORK_DIR / f"fold{fold + 1}",  # チェックポイント保存先
        report_to="none",  # TensorBoardなどへの送信なし
        max_steps=config.max_steps,  # 最大1000ステップ
        
        ### バッチサイズ設定
        per_device_train_batch_size=config.per_device_train_batch_size,  # 物理バッチ: 8
        gradient_accumulation_steps=config.gradient_accumulation_steps,  # 勾配累積: 8
        # 実質バッチサイズ = 8 × 8 = 64
        per_device_eval_batch_size=config.per_device_eval_batch_size,  # 評価時: 16
        
        ### 実行モード
        do_train=True,  # 学習を実行
        do_eval=True,  # 評価を実行
        do_predict=True,  # 予測を実行
        
        ### ログ・評価・保存の頻度
        logging_strategy="steps",
        eval_strategy="steps",
        save_strategy="steps",
        logging_steps=config.eval_steps,  # 50ステップごとにログ
        eval_steps=config.eval_steps,  # 50ステップごとに評価
        save_steps=config.eval_steps,  # 50ステップごとに保存
        save_total_limit=1,  # 最新のチェックポイント1つのみ保持
        
        ### ベストモデルの選択（コメントアウト）
        # load_best_model_at_end=True,  # 最後にベストモデルをロード
        # metric_for_best_model="eval_loss",  # 評価損失でベストを判定
        # greater_is_better=False,  # 小さいほど良い
        
        ### 最適化設定
        optim=config.optim_type,  # adamw_torch
        bf16=True,  # BFloat16混合精度学習
        learning_rate=config.lr,  # 2.0e-5
        weight_decay=config.weight_decay,  # L2正則化: 0.01
        warmup_steps=config.warmup_steps,  # ウォームアップ: 25ステップ
        lr_scheduler_type=config.lr_scheduler_type,  # cosine
        max_grad_norm=config.max_grad_norm,  # 勾配クリッピング: 10.0
        
        ### 乱数シード
        seed=config.seed,
        data_seed=config.seed,
    )
    
    ## Trainerの作成
    trainer = Trainer(
        args=training_args,
        model=model,
        train_dataset=tr_ds,  # 学習データ
        eval_dataset=val_ds,  # 検証データ
        compute_metrics=compute_metrics,  # ROC-AUC計算関数
        data_collator=DataCollatorWithPadding(tokenizer),  # 動的パディング
    )
    
    ## 学習実行
    trainer.train()

    ## モデルの保存
    model_path = WORK_DIR / f"model_fold{fold + 1}"
    trainer.save_model(model_path)

    ## 中間チェックポイントの削除（ディスク容量節約）
    fold_dir = WORK_DIR / f"fold{fold + 1}"
    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    ## OOF予測の作成
    # logitsをSoftmaxで確率に変換し、クラス1（正例）の確率を取得
    probs = (
        torch.from_numpy(trainer.predict(val_ds).predictions)
        .float()
        .softmax(-1)  # Softmax適用
        .numpy()[:, 1]  # クラス1の確率
    )
    # 検証データに予測確率を追加
    oof_df = val_ds.to_polars().with_columns(
        pl.Series("prediction", probs, dtype=pl.Float32)
    )
    oof_predictions.append(oof_df)

    ## Foldごとのスコア計算
    fold_score = roc_auc_score(
        oof_df.get_column("labels").to_numpy(),
        oof_df.get_column("prediction").to_numpy(),
    )
    print(f"Fold {fold + 1} AUC: {fold_score:.5f}")

## 全Foldの予測を結合
oof_prediction_df = pl.concat(oof_predictions, how="vertical").sort(
    "employee_id", "category"
)
# OOF予測を保存
oof_prediction_df.select(
    "employee_id", "category", "fold", "labels", "prediction"
).write_csv(WORK_DIR / "oof_predictions.csv")

## 全体のCVスコア計算
valid_score = roc_auc_score(
    oof_prediction_df.get_column("labels").to_numpy(),
    oof_prediction_df.get_column("prediction").to_numpy(),
)
print(f"valid_score\t:\t {valid_score:.5f}")

In [ ]:
# ============================================
# テストデータの前処理
# ============================================

## パス設定
DATA_DIR = Path(config.input_dir)
WORK_DIR = Path(config.work_dir)

## テストデータ読み込み
test_df = pl.read_csv(DATA_DIR / "test.csv").with_columns(
    pl.lit(-1, dtype=pl.Int64).alias("target"),  # ダミーのtarget（前処理関数で必要）
    pl.lit(-1, dtype=pl.Int64).alias("fold"),  # ダミーのfold
)
# 研修・勤怠データ読み込み（trainと同じ）
dx_df = pl.read_csv(DATA_DIR / "dx.csv", schema=DX_SCHEMA)
hr_df = pl.read_csv(DATA_DIR / "hr.csv", schema=HR_SCHEMA)
overtime_work_by_month_df = pl.read_csv(
    DATA_DIR / "overtime_work_by_month.csv", schema=OVERTIME_WORK_BY_MONTH_SCHEMA
)
udemy_activity_df = pl.read_csv(
    DATA_DIR / "udemy_activity.csv", schema=UDEMY_ACTIVITY_SCHEMA
)
position_history_df = pl.read_csv(DATA_DIR / "position_history.csv")

## データ前処理（年度カラムの追加）
(udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df) = (
    preprocess_data(
        udemy_activity_df, dx_df, hr_df, overtime_work_by_month_df, position_history_df
    )
)

## トークナイザ読み込み
tokenizer = AutoTokenizer.from_pretrained(WORK_DIR / "tokenizer")

## プロンプト作成（テーブルデータ→テキスト変換）
test_data = process_data(
    test_df,
    udemy_activity_df,
    dx_df,
    hr_df,
    overtime_work_by_month_df,
    position_history_df,
)
test_data.write_csv(WORK_DIR / "test_data.csv")

## データセット変換とトークン化
test_ds = Dataset.from_polars(test_data)
test_ds = test_ds.map(
    lambda batch: encode_prompt(batch, tokenizer, max_length=config.max_length),
    batched=True,
    remove_columns=["text"],
).remove_columns(["fold", "labels"])  # テストデータには不要なカラムを削除


In [ ]:
# ============================================
# テストデータ予測とアンサンブル
# ============================================
test_predictions = []  # 各Foldの予測を格納するリスト

## 各Foldのモデルで予測
for fold in range(config.n_splits):
    print(f"### Predicting Fold {fold + 1}/{config.n_splits} ###")

    ### 学習済みモデルの読み込み
    model = ModernBertClassifier.from_pretrained(
        WORK_DIR / f"model_fold{fold + 1}",  # 保存したモデルを読み込み
        classifier_pooling=config.classifier_pooling,
        num_labels=2,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    
    ### 予測用の設定
    predict_args = TrainingArguments(
        report_to="none",
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        bf16=True,  # BFloat16で高速化
        optim=config.optim_type,
        seed=config.seed,
        data_seed=config.seed,
        do_train=False,  # 学習しない
        do_eval=False,  # 評価しない
        do_predict=True,  # 予測のみ実行
    )
    
    ### 予測用Trainerの作成
    predict_trainer = Trainer(
        args=predict_args, 
        model=model, 
        data_collator=DataCollatorWithPadding(tokenizer)
    )

    ### 予測実行
    # logitsをSoftmaxで確率に変換し、クラス1の確率を取得
    probs = (
        torch.from_numpy(predict_trainer.predict(test_ds).predictions)
        .float()
        .softmax(-1)  # Softmax適用
        .numpy()[:, 1]  # クラス1（関心あり）の確率
    )
    
    # 予測結果をDataFrameに追加
    pred_df = test_ds.to_polars().with_columns(
        pl.lit(fold, dtype=pl.Int64).alias("fold"),  # Fold番号
        pl.Series("prediction", probs, dtype=pl.Float32),  # 予測確率
    )
    test_predictions.append(pred_df)

## テスト予測の保存
# 全Foldの予測を結合
test_prediction_df = pl.concat(test_predictions, how="vertical").select(
    "employee_id", "category", "fold", "prediction"
)

## アンサンブル（3Foldの平均）
submission_df = (
    test_prediction_df.group_by("employee_id", "category")
    .agg(pl.col("prediction").mean())  # 3つのFoldの予測確率を平均
    .select("employee_id", "category", "prediction")
    .sort("employee_id", "category")  # 提出用にソート
    .select(pl.col("prediction").alias("target"))  # カラム名を"target"に変更
)

## 保存
test_prediction_df.write_csv(WORK_DIR / "test_predictions.csv")  # Fold別予測
submission_df.write_csv(WORK_DIR / f"{config.exp_name}.csv")  # 最終提出ファイル

In [ ]:
# test prediction
test_predictions = []
for fold in range(config.n_splits):
    print(f"### Predicting Fold {fold + 1}/{config.n_splits} ###")

    model = ModernBertClassifier.from_pretrained(
        WORK_DIR / f"model_fold{fold + 1}",
        classifier_pooling=config.classifier_pooling,
        num_labels=2,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    predict_args = TrainingArguments(
        report_to="none",
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        bf16=True,
        optim=config.optim_type,
        seed=config.seed,
        data_seed=config.seed,
        do_train=False,
        do_eval=False,
        do_predict=True,
    )
    predict_trainer = Trainer(
        args=predict_args, model=model, data_collator=DataCollatorWithPadding(tokenizer)
    )

    probs = (
        torch.from_numpy(predict_trainer.predict(test_ds).predictions)
        .float()
        .softmax(-1)
        .numpy()[:, 1]
    )
    pred_df = test_ds.to_polars().with_columns(
        pl.lit(fold, dtype=pl.Int64).alias("fold"),
        pl.Series("prediction", probs, dtype=pl.Float32),
    )
    test_predictions.append(pred_df)

# save test predictions
test_prediction_df = pl.concat(test_predictions, how="vertical").select(
    "employee_id", "category", "fold", "prediction"
)

submission_df = (
    test_prediction_df.group_by("employee_id", "category")
    .agg(pl.col("prediction").mean())
    .select("employee_id", "category", "prediction")
    .sort("employee_id", "category")
    .select(pl.col("prediction").alias("target"))
)

test_prediction_df.write_csv(WORK_DIR / "test_predictions.csv")
submission_df.write_csv(WORK_DIR / f"{config.exp_name}.csv")